# Credit Risk Monitoring Report

This notebook is the editable version of the PDF generator now stored in `src/credit_report.py`.
It keeps the same ReportLab workflow, but breaks the logic into smaller sections so the project is easier to read on GitHub and easier to update later.

Edit these blocks most often:
- `REPORT_CONFIG` for dates, author details, and cover-page metadata
- `PORTFOLIO` for the monitored entity table
- `WATCHLIST_ENTRIES` for the watchlist cards
- `DETAILED_NOTES` for the long-form credit notes

Run the notebook from top to bottom, then execute the final cell to regenerate `reports/ThanhPham_CreditRisk_Report.pdf`.
If ReportLab is missing, install it with `pip install reportlab`.


## 1. Imports and document geometry

This cell loads the ReportLab building blocks and defines the page size, margins, and output path.
Keep these settings together so layout changes stay easy to trace.


In [2]:
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT, TA_RIGHT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    HRFlowable,
    KeepTogether,
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)
from reportlab.platypus.flowables import Flowable

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = REPORTS_DIR / "ThanhPham_CreditRisk_Report.pdf"

W, H = A4
MARGIN_L = 2.0 * cm
MARGIN_R = 2.0 * cm
MARGIN_T = 1.2 * cm
MARGIN_B = 1.6 * cm
CONTENT_W = W - MARGIN_L - MARGIN_R


ModuleNotFoundError: No module named 'reportlab'

## 2. Visual configuration

The report uses a small palette and a few shared lookup tables.
Changing the values below is the fastest way to restyle the full document consistently.


In [ ]:
NAVY = HexColor("#0B1F3A")
NAVY_LIGHT = HexColor("#1A3560")
STEEL = HexColor("#2C4A7C")
SILVER = HexColor("#F4F6FA")
MID_GREY = HexColor("#8A97A8")
LINE_GREY = HexColor("#D0D8E4")
BODY_TEXT = HexColor("#1A1A2E")
WHITE = colors.white

RED_ALERT = HexColor("#C0392B")
AMBER_ALERT = HexColor("#E67E22")
GREEN_OK = HexColor("#1E8449")
BLUE_NOTE = HexColor("#1A6FA5")
GOLD = HexColor("#B8860B")

RATING_COLOR = {
    "AA": HexColor("#1E5799"),
    "A": BLUE_NOTE,
    "BBB": HexColor("#2980B9"),
    "BB": GOLD,
    "B": AMBER_ALERT,
    "CCC": RED_ALERT,
}

ACTION_COLOR = {
    "Monitor": GREEN_OK,
    "Review": AMBER_ALERT,
    "Escalate": RED_ALERT,
}


## 3. Editable report content

This is the main business-content section.
The rest of the notebook mostly controls presentation, so if you need to refresh the report for a new date or committee pack, start here.


In [ ]:
REPORT_CONFIG = {
    "reference_date": "31 December 2025",
    "classification": "Internal - Restricted",
    "portfolio_scope": "25 Entities (15 Banks / 10 Sovereigns)",
    "prepared_by": "Thanh Pham | Credit Risk Analyst",
    "distribution": "CRO, Risk Committee, Clearing Members Desk",
    "committee_banner": "INTERNAL - CREDIT RISK COMMITTEE USE ONLY",
    "division_line": "GCH Credit & Counterparty Risk Division",
    "footer_title": "Credit Risk Monitoring Report",
}

PORTFOLIO_OUTLOOK = {
    "label": "STABLE",
    "note": "Selective downside risk",
}

EXECUTIVE_SUMMARY_BULLETS = [
    "<b>Portfolio scope:</b> 25 counterparties monitored (15 banks, 10 sovereigns). 18 entities are currently on the watchlist (72% of portfolio), reflecting elevated macro uncertainty and persistent spread widening across European credit.",
    "<b>Tail risk concentration:</b> One entity - Monte dei Paschi (Bank, Italy) - has been escalated for immediate committee action. CDS at 306 bp, negative ROE, and a CCC internal rating signal material credit deterioration requiring position review.",
    "<b>Sub-investment-grade dominance:</b> Only 4 of 25 entities score at BBB or above under the internal model (Nordea, JPMorgan Chase, ING Group, Intesa Sanpaolo). The remaining 21 carry sub-IG internal ratings - 11 rated B, 9 rated BB, and 1 CCC.",
    "<b>Sovereign stress is systemic, not idiosyncratic:</b> Eight of ten sovereigns are watchlisted. Key constraints include high debt/GDP ratios (Italy 144%, Greece 185%), weak fiscal balances, and limited FX reserve buffers. Germany remains the sole sovereign off-watchlist.",
    "<b>Market signals confirm credit stress:</b> CDS-bond spread correlation stands at 0.98 (30-day). Watchlist entities trade at an average CDS of 135 bp vs. 68 bp for non-watchlist - a 2x pricing differential validating internal risk segmentation. Turkey (473 bp) and Brazil (281 bp) represent the widest EM sovereign exposures.",
    "<b>Overall credit outlook:</b> Stable with selective downside risk. No systemic stress event is imminent. Risk remains concentrated in specific names rather than portfolio-wide. Primary watch-points: Italian sovereign trajectory, French bank profitability, and Monte dei Paschi balance sheet repair.",
]

PORTFOLIO = [
    {"name": "JPMorgan Chase", "entity_type": "Bank", "score": 71.0, "rating": "BBB", "alerts": 1, "watchlist": "No", "cds": 44.1, "action": "Monitor", "highlight": False},
    {"name": "HSBC", "entity_type": "Bank", "score": 59.0, "rating": "BB", "alerts": 2, "watchlist": "Yes", "cds": 58.3, "action": "Review", "highlight": False},
    {"name": "BNP Paribas", "entity_type": "Bank", "score": 54.3, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 74.2, "action": "Review", "highlight": False},
    {"name": "Deutsche Bank", "entity_type": "Bank", "score": 51.3, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 123.2, "action": "Review", "highlight": True},
    {"name": "UniCredit", "entity_type": "Bank", "score": 63.2, "rating": "BB", "alerts": 0, "watchlist": "No", "cds": 61.3, "action": "Monitor", "highlight": False},
    {"name": "Santander", "entity_type": "Bank", "score": 59.6, "rating": "BB", "alerts": 1, "watchlist": "Yes", "cds": 86.7, "action": "Review", "highlight": False},
    {"name": "ING Group", "entity_type": "Bank", "score": 65.3, "rating": "BBB", "alerts": 0, "watchlist": "No", "cds": 41.2, "action": "Monitor", "highlight": False},
    {"name": "Societe Generale", "entity_type": "Bank", "score": 48.0, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 129.2, "action": "Review", "highlight": True},
    {"name": "Barclays", "entity_type": "Bank", "score": 58.5, "rating": "BB", "alerts": 1, "watchlist": "Yes", "cds": 78.2, "action": "Review", "highlight": False},
    {"name": "UBS", "entity_type": "Bank", "score": 64.3, "rating": "BB", "alerts": 0, "watchlist": "No", "cds": 38.5, "action": "Monitor", "highlight": False},
    {"name": "Nordea", "entity_type": "Bank", "score": 76.2, "rating": "A", "alerts": 0, "watchlist": "No", "cds": 21.3, "action": "Monitor", "highlight": False},
    {"name": "Intesa Sanpaolo", "entity_type": "Bank", "score": 68.1, "rating": "BBB", "alerts": 0, "watchlist": "No", "cds": 55.7, "action": "Monitor", "highlight": False},
    {"name": "ABN AMRO", "entity_type": "Bank", "score": 60.0, "rating": "BB", "alerts": 1, "watchlist": "Yes", "cds": 70.6, "action": "Review", "highlight": False},
    {"name": "Commerzbank", "entity_type": "Bank", "score": 55.8, "rating": "BB", "alerts": 2, "watchlist": "Yes", "cds": 111.6, "action": "Review", "highlight": True},
    {"name": "Monte dei Paschi", "entity_type": "Bank", "score": 36.8, "rating": "CCC", "alerts": 4, "watchlist": "Yes", "cds": 306.1, "action": "Escalate", "highlight": True},
    {"name": "Germany", "entity_type": "Sovereign", "score": 61.3, "rating": "BB", "alerts": 0, "watchlist": "No", "cds": 12.1, "action": "Monitor", "highlight": False},
    {"name": "France", "entity_type": "Sovereign", "score": 45.4, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 39.9, "action": "Review", "highlight": False},
    {"name": "United States", "entity_type": "Sovereign", "score": 40.1, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 29.7, "action": "Review", "highlight": False},
    {"name": "United Kingdom", "entity_type": "Sovereign", "score": 46.1, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 37.4, "action": "Review", "highlight": False},
    {"name": "Japan", "entity_type": "Sovereign", "score": 54.1, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 19.4, "action": "Review", "highlight": False},
    {"name": "Italy", "entity_type": "Sovereign", "score": 40.6, "rating": "B", "alerts": 2, "watchlist": "Yes", "cds": 147.7, "action": "Review", "highlight": True},
    {"name": "Spain", "entity_type": "Sovereign", "score": 50.2, "rating": "B", "alerts": 1, "watchlist": "Yes", "cds": 88.6, "action": "Review", "highlight": False},
    {"name": "Greece", "entity_type": "Sovereign", "score": 42.4, "rating": "B", "alerts": 2, "watchlist": "Yes", "cds": 271.5, "action": "Review", "highlight": True},
    {"name": "Turkey", "entity_type": "Sovereign", "score": 46.9, "rating": "B", "alerts": 3, "watchlist": "Yes", "cds": 472.9, "action": "Review", "highlight": True},
    {"name": "Brazil", "entity_type": "Sovereign", "score": 56.5, "rating": "BB", "alerts": 2, "watchlist": "Yes", "cds": 281.3, "action": "Review", "highlight": True},
]

WATCHLIST_INTRO = (
    "18 entities meet at least one watchlist criterion: (i) internal score below 60, "
    "(ii) non-investment-grade external rating, (iii) two or more active market alerts, "
    "or (iv) score deterioration exceeding 10 points. The five highest-severity names are "
    "detailed below. Remaining 13 names are on active monitor status."
)

WATCHLIST_ENTRIES = [
    {
        "name": "Monte dei Paschi",
        "type": "Bank",
        "score": 36.8,
        "rating": "CCC",
        "reason": "NPL ratio 12.4%, negative ROE (-2.5%), and cost-to-income at 88%. Internal score breached the CCC threshold. CDS at 306 bp and equity volatility above 46% confirm sustained market distress.",
        "cds": "306.1 bp",
        "spread": "342.1 bp",
        "equity_5d": "+0.6%",
        "vol": "46.1%",
        "action": "Escalate",
    },
    {
        "name": "Turkey",
        "type": "Sovereign",
        "score": 46.9,
        "rating": "B",
        "reason": "Inflation at 48%, political risk 7/10, and limited FX buffers. CDS at 472.9 bp is the widest in the portfolio, while equity volatility at 53.5% reflects persistent investor risk aversion.",
        "cds": "472.9 bp",
        "spread": "402.1 bp",
        "equity_5d": "-1.8%",
        "vol": "53.5%",
        "action": "Review",
    },
    {
        "name": "Greece",
        "type": "Sovereign",
        "score": 42.4,
        "rating": "B",
        "reason": "Debt-to-GDP at 185% with an active bond spread widening alert at 271.5 bp. The score remains in the lower-B range with no obvious near-term upside drivers.",
        "cds": "271.5 bp",
        "spread": "283.4 bp",
        "equity_5d": "-2.1%",
        "vol": "34.8%",
        "action": "Review",
    },
    {
        "name": "Italy",
        "type": "Sovereign",
        "score": 40.6,
        "rating": "B",
        "reason": "Debt-to-GDP at 144%, a structural fiscal deficit, and a 5-day equity move of -5.3%. CDS at 148 bp still signals fragile sovereign sentiment and limited rating headroom.",
        "cds": "147.7 bp",
        "spread": "158.2 bp",
        "equity_5d": "-5.3%",
        "vol": "31.2%",
        "action": "Review",
    },
    {
        "name": "Societe Generale",
        "type": "Bank",
        "score": 48.0,
        "rating": "B",
        "reason": "Internal B versus external A- reflects weaker liquidity and efficiency metrics. CDS at 129 bp aligns more closely with the internal view than with the external rating anchor.",
        "cds": "129.2 bp",
        "spread": "122.5 bp",
        "equity_5d": "+0.3%",
        "vol": "26.3%",
        "action": "Review",
    },
]

DETAILED_NOTES = [
    {
        "name": "Monte dei Paschi di Siena",
        "tag": "Bank | Italy | CCC",
        "tag_color": RED_ALERT,
        "overview": "Italy's third-largest bank by assets. Despite a 2022 state recapitalisation, structural weaknesses persist: negative profitability, legacy NPLs, and cost rigidity. It is the sole 'Escalate' entity in the current monitoring cycle.",
        "strengths": [
            "CET1 ratio 18.3% - well above regulatory minima, providing a capital buffer.",
            "Loan-to-deposit ratio 0.95x - balanced funding with no immediate liquidity cliff.",
            "State backstop preserves systemic relevance and lowers short-term insolvency risk.",
        ],
        "weaknesses": [
            "NPL ratio 12.4% - highest in the bank universe and roughly 4x the portfolio median.",
            "Negative ROE (-2.5%) and cost-to-income at 88.4% point to structural unprofitability.",
            "CDS at 306 bp and equity volatility at 46.1% show persistent market scepticism.",
        ],
        "signals": [
            ("CDS 5Y", "306.1 bp", RED_ALERT),
            ("Bond Spread", "342.1 bp", RED_ALERT),
            ("Equity 5d Return", "+0.6%", GREEN_OK),
            ("Implied Vol (30d)", "46.1%", RED_ALERT),
        ],
        "int_rating": "CCC",
        "ext_rating": "B",
        "outlook": "Negative",
        "analyst_view": "MPS is the portfolio's highest-severity name. The CCC rating reflects genuine fundamental impairment. Recommend review of exposure limits and collateral terms. Escalation to default-watch should trigger if NPLs exceed 15%, CDS remains above 400 bp, or a regulatory capital call emerges.",
    },
    {
        "name": "Republic of Turkey",
        "tag": "Sovereign | EM | B",
        "tag_color": AMBER_ALERT,
        "overview": "Widest CDS in the portfolio at 472.9 bp. Structurally high inflation, political risk, and limited monetary policy credibility are the main drivers. The internal model assigns B, broadly in line with the external B+ anchor.",
        "strengths": [
            "GDP growth at 4.2% remains above the EM peer average, providing a cyclical buffer.",
            "FX reserves have stabilised since the 2023 policy normalisation phase.",
            "The rate-hike cycle signals partial restoration of central bank credibility.",
        ],
        "weaknesses": [
            "Inflation at 48% is the highest in the sovereign portfolio and the main score drag.",
            "Political risk score 7/10 keeps institutional stability concerns elevated.",
            "CDS at 472.9 bp and equity volatility at 53.5% are the highest in the portfolio.",
        ],
        "signals": [
            ("CDS 5Y", "472.9 bp", RED_ALERT),
            ("Bond Spread", "402.1 bp", RED_ALERT),
            ("Equity 5d Return", "-1.8%", AMBER_ALERT),
            ("Implied Vol (30d)", "53.5%", RED_ALERT),
        ],
        "int_rating": "B",
        "ext_rating": "B+",
        "outlook": "Stable / Watchlisted",
        "analyst_view": "Turkey's risk profile is still driven by macro variables that will not normalise quickly. A meaningful reassessment would require inflation to fall below 30%. Collateral margins should remain aligned with current CDS levels, and renewed lira weakness or political stress would justify escalation.",
    },
]

MARKET_MONITORING_ROWS = [
    (
        "CDS Trends",
        "Elevated but range-bound across Q4 2025, with no systemic widening event. Turkey (473 bp) and Monte dei Paschi (306 bp) remained stressed throughout. Roughly 48 CDS alerts over 90 days were concentrated in two short episodes rather than a broad market shock.",
    ),
    (
        "Bond Spreads",
        "CDS-bond spread correlation stands at 0.98, showing that stress is expressed almost entirely through credit markets. Greece is the only active spread alert. Peripheral spreads for Italy, Greece, and Spain remain consistent with lower-quality sovereign profiles.",
    ),
    (
        "Equity Volatility",
        "Equity-volatility-to-CDS correlation is 0.82. Four entities remain above the 35% implied volatility trigger: Turkey, Monte dei Paschi, Brazil, and Commerzbank. Core developed-market bank volatility has otherwise normalised.",
    ),
    (
        "Stress Assessment",
        "Systemic risk is low, while idiosyncratic risk is elevated for weaker names. Watchlist entities trade around 2x wider than non-watchlist peers on CDS, which supports the internal model's discriminatory power.",
    ),
]

METHODOLOGY_BULLETS = [
    "<b>Internal scoring (Banks):</b> A 7-factor weighted model on a 0-100 scale combining CET1 ratio (25%), NPL ratio (20%), ROE (15%), leverage (10%), loan-to-deposit (10%), cost-to-income (10%), and liquidity ratio (10%).",
    "<b>Internal scoring (Sovereigns):</b> Seven macro-fiscal factors - debt/GDP (25%), fiscal balance (20%), inflation (15%), FX reserves (15%), GDP growth (10%), political risk (10%), and current account (5%).",
    "<b>Rating mapping:</b> Score ranges map to AA (85-100), A (75-84), BBB (65-74), BB (55-64), B (40-54), and CCC (below 40).",
    "<b>Market alerts:</b> Four independent triggers - CDS widening above 20 bp over 5 days, bond spread widening above 25 bp over 5 days, 5-day equity return below -10%, and 30-day implied equity volatility above 35%.",
    "<b>Watchlist logic:</b> An entity is watchlisted if it breaches at least one threshold: internal score below 60, non-investment-grade external rating, two or more concurrent market alerts, or score deterioration above 10 points versus the prior snapshot.",
    "<b>Limitations:</b> The framework is designed for relative ranking inside a clearing-house context, not for direct default-probability estimation. Final decisions should still incorporate qualitative judgement and external intelligence.",
]

DISCLAIMER_TEXT = (
    "This report was prepared by Thanh Pham, Credit Risk Analyst. All ratings, scores, and "
    "market signals are based on the internal credit risk monitoring framework as of 31 December 2025. "
    "Internal ratings do not constitute investment advice and are prepared for risk-management purposes only. "
    "Reproduction or external distribution is prohibited."
)


## 4. Shared styles and small helpers

These helpers keep the section-building code shorter.
The goal is to make later cells read more like report assembly than low-level drawing logic.


In [ ]:
def build_styles():
    def style(name, **kwargs):
        return ParagraphStyle(name, **kwargs)

    return {
        "cover_title": style(
            "cover_title",
            fontName="Helvetica-Bold",
            fontSize=22,
            textColor=WHITE,
            leading=28,
            alignment=TA_LEFT,
        ),
        "cover_sub": style(
            "cover_sub",
            fontName="Helvetica",
            fontSize=11,
            textColor=HexColor("#A8C0DC"),
            leading=16,
            alignment=TA_LEFT,
        ),
        "cover_meta": style(
            "cover_meta",
            fontName="Helvetica",
            fontSize=9,
            textColor=HexColor("#C8D8EC"),
            leading=13,
            alignment=TA_LEFT,
        ),
        "section_header": style(
            "section_header",
            fontName="Helvetica-Bold",
            fontSize=11,
            textColor=WHITE,
            leading=14,
            alignment=TA_LEFT,
        ),
        "subsection": style(
            "subsection",
            fontName="Helvetica-Bold",
            fontSize=9.5,
            textColor=NAVY,
            leading=13,
            spaceBefore=8,
            spaceAfter=3,
        ),
        "body": style(
            "body",
            fontName="Helvetica",
            fontSize=8,
            textColor=BODY_TEXT,
            leading=12,
            alignment=TA_JUSTIFY,
            spaceAfter=3,
        ),
        "bullet": style(
            "bullet",
            fontName="Helvetica",
            fontSize=8,
            textColor=BODY_TEXT,
            leading=12,
            leftIndent=12,
            bulletIndent=0,
            spaceAfter=2,
        ),
        "table_header": style(
            "table_header",
            fontName="Helvetica-Bold",
            fontSize=8,
            textColor=WHITE,
            leading=11,
            alignment=TA_CENTER,
        ),
        "table_cell": style(
            "table_cell",
            fontName="Helvetica",
            fontSize=8,
            textColor=BODY_TEXT,
            leading=11,
            alignment=TA_LEFT,
        ),
        "table_cell_c": style(
            "table_cell_c",
            fontName="Helvetica",
            fontSize=8,
            textColor=BODY_TEXT,
            leading=11,
            alignment=TA_CENTER,
        ),
        "table_cell_bold": style(
            "table_cell_bold",
            fontName="Helvetica-Bold",
            fontSize=8,
            textColor=NAVY,
            leading=11,
            alignment=TA_LEFT,
        ),
        "caption": style(
            "caption",
            fontName="Helvetica-Oblique",
            fontSize=7.5,
            textColor=MID_GREY,
            leading=10,
            alignment=TA_CENTER,
            spaceAfter=6,
        ),
        "disclaimer": style(
            "disclaimer",
            fontName="Helvetica-Oblique",
            fontSize=7,
            textColor=MID_GREY,
            leading=10,
            alignment=TA_LEFT,
        ),
    }


class SectionHeader(Flowable):
    def __init__(self, number, title, width):
        super().__init__()
        self.number = number
        self.title = title
        self.width = width
        self.height = 0.65 * cm

    def draw(self):
        self.canv.setFillColor(STEEL)
        self.canv.rect(0, 0, self.width, self.height, fill=1, stroke=0)
        self.canv.setFillColor(WHITE)
        self.canv.setFont("Helvetica-Bold", 10)
        self.canv.drawString(0.3 * cm, 0.18 * cm, f"  {self.number}  {self.title.upper()}")

    def wrap(self, *args):
        return self.width, self.height + 4


def hr(width=CONTENT_W, thickness=0.5, color=LINE_GREY):
    return HRFlowable(width=width, thickness=thickness, color=color, spaceBefore=4, spaceAfter=4)


def sp(height=4):
    return Spacer(1, height)


def bullet_para(text, styles):
    return Paragraph(f"<bullet>&bull;</bullet> {text}", styles["bullet"])


## 5. Page template

The first page is drawn directly on the canvas to give the cover a cleaner, presentation-style layout.
Later pages use a lighter header and footer frame.


In [ ]:
def draw_page_frame(canvas_obj, doc):
    canvas_obj.saveState()

    canvas_obj.setFillColor(NAVY)
    canvas_obj.rect(0, H - 0.55 * cm, W, 0.55 * cm, fill=1, stroke=0)
    canvas_obj.setFillColor(WHITE)
    canvas_obj.setFont("Helvetica-Bold", 7)
    canvas_obj.drawString(MARGIN_L, H - 0.38 * cm, REPORT_CONFIG["committee_banner"])
    canvas_obj.setFont("Helvetica", 7)
    canvas_obj.drawRightString(W - MARGIN_R, H - 0.38 * cm, REPORT_CONFIG["division_line"])

    canvas_obj.setStrokeColor(LINE_GREY)
    canvas_obj.setLineWidth(0.5)
    canvas_obj.line(MARGIN_L, MARGIN_B - 0.2 * cm, W - MARGIN_R, MARGIN_B - 0.2 * cm)
    canvas_obj.setFillColor(MID_GREY)
    canvas_obj.setFont("Helvetica", 7)
    canvas_obj.drawString(
        MARGIN_L,
        MARGIN_B - 0.45 * cm,
        f"{REPORT_CONFIG['footer_title']} | Ref. Date: {REPORT_CONFIG['reference_date']}",
    )
    canvas_obj.drawRightString(W - MARGIN_R, MARGIN_B - 0.45 * cm, f"Page {doc.page}")
    canvas_obj.restoreState()


def draw_cover_page(canvas_obj, doc):
    c = canvas_obj
    c.saveState()

    c.setFillColor(NAVY)
    c.rect(0, 0, W, H, fill=1, stroke=0)

    c.setFillColor(HexColor("#1F6AA5"))
    c.rect(0, 0, 0.6 * cm, H, fill=1, stroke=0)
    c.rect(0.6 * cm, H - 0.4 * cm, W - 0.6 * cm, 0.4 * cm, fill=1, stroke=0)

    c.setFillColor(HexColor("#A8C0DC"))
    c.setFont("Helvetica", 7.5)
    c.drawString(2.2 * cm, H - 0.28 * cm, REPORT_CONFIG["committee_banner"])

    c.setFont("Helvetica", 9)
    c.drawString(2.2 * cm, H - 3.2 * cm, "GLOBAL CLEARING HOUSE | CREDIT & COUNTERPARTY RISK DIVISION")

    c.setFillColor(WHITE)
    c.setFont("Helvetica-Bold", 28)
    c.drawString(2.2 * cm, H - 6.0 * cm, "Credit Risk Monitoring")
    c.setFont("Helvetica-Bold", 22)
    c.drawString(2.2 * cm, H - 7.6 * cm, "Committee Report")

    c.setFillColor(HexColor("#D4A017"))
    c.rect(2.2 * cm, H - 8.2 * cm, 9 * cm, 0.08 * cm, fill=1, stroke=0)
    c.rect(0.6 * cm, H * 0.48, W - 0.6 * cm, 0.06 * cm, fill=1, stroke=0)

    meta_rows = [
        ("Reference Date", REPORT_CONFIG["reference_date"]),
        ("Report Classification", REPORT_CONFIG["classification"]),
        ("Portfolio Scope", REPORT_CONFIG["portfolio_scope"]),
        ("Prepared by", REPORT_CONFIG["prepared_by"]),
        ("Distribution", REPORT_CONFIG["distribution"]),
    ]

    y = H * 0.44
    for label, value in meta_rows:
        c.setFillColor(HexColor("#122D52"))
        c.rect(2.2 * cm, y - 0.05 * cm, 4.5 * cm, 0.55 * cm, fill=1, stroke=0)
        c.setFillColor(HexColor("#8AAFC8"))
        c.setFont("Helvetica", 7.5)
        c.drawString(2.4 * cm, y + 0.08 * cm, label.upper())
        c.setFillColor(WHITE)
        c.setFont("Helvetica-Bold", 9)
        c.drawString(7.2 * cm, y + 0.08 * cm, value)
        y -= 0.72 * cm

    c.setFillColor(HexColor("#4A6B8A"))
    c.setFont("Helvetica-Oblique", 7.5)
    c.drawString(
        2.2 * cm,
        2.2 * cm,
        "This document is prepared for internal risk committee use only and must not be distributed externally.",
    )
    c.drawString(
        2.2 * cm,
        1.55 * cm,
        "Ratings and assessments reflect internal model outputs as of the reference date and are subject to revision.",
    )
    c.restoreState()


## 6. Section renderers

These functions handle the heavier layout pieces such as the portfolio table, watchlist cards, and detailed credit notes.
Keeping them separate makes the final report builder much easier to scan.


In [ ]:
def build_outlook_table():
    label_style = ParagraphStyle(
        "outlook_label",
        fontName="Helvetica-Bold",
        fontSize=9,
        textColor=NAVY,
        alignment=TA_CENTER,
    )
    value_style = ParagraphStyle(
        "outlook_value",
        fontName="Helvetica-Bold",
        fontSize=10,
        textColor=GREEN_OK,
        alignment=TA_CENTER,
    )
    table = Table(
        [[
            Paragraph("<b>OVERALL PORTFOLIO OUTLOOK</b>", label_style),
            Paragraph(
                f"<b>{PORTFOLIO_OUTLOOK['label']}</b> - {PORTFOLIO_OUTLOOK['note']}",
                value_style,
            ),
        ]],
        colWidths=[7 * cm, 9 * cm],
    )
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, 0), SILVER),
        ("BACKGROUND", (1, 0), (1, 0), HexColor("#EAF6EE")),
        ("BOX", (0, 0), (-1, -1), 1.0, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))
    return table


def build_portfolio_table(styles):
    headers = ["Entity", "Type", "Score", "Int. Rating", "CDS (bp)", "Alerts", "Watchlist"]
    rows = [[Paragraph(header, styles["table_header"]) for header in headers]]
    sorted_portfolio = sorted(PORTFOLIO, key=lambda item: item["score"])

    for item in sorted_portfolio:
        rating_col = RATING_COLOR.get(item["rating"], MID_GREY)
        watchlist_col = RED_ALERT if item["watchlist"] == "Yes" else GREEN_OK
        alert_col = RED_ALERT if item["alerts"] >= 3 else AMBER_ALERT if item["alerts"] >= 2 else NAVY
        score_col = RED_ALERT if item["score"] < 40 else AMBER_ALERT if item["score"] < 55 else NAVY

        rows.append([
            Paragraph(
                f"<b>{item['name']}</b>" if item["highlight"] else item["name"],
                styles["table_cell_bold"] if item["highlight"] else styles["table_cell"],
            ),
            Paragraph(item["entity_type"], styles["table_cell_c"]),
            Paragraph(
                f"<b>{item['score']:.1f}</b>",
                ParagraphStyle(
                    f"score_{item['name']}",
                    fontName="Helvetica-Bold" if item["score"] < 45 else "Helvetica",
                    fontSize=8,
                    textColor=score_col,
                    alignment=TA_CENTER,
                ),
            ),
            Paragraph(
                f"<b>{item['rating']}</b>",
                ParagraphStyle(
                    f"rating_{item['name']}",
                    fontName="Helvetica-Bold",
                    fontSize=8,
                    textColor=rating_col,
                    alignment=TA_CENTER,
                ),
            ),
            Paragraph(
                f"{item['cds']:.1f}",
                ParagraphStyle(
                    f"cds_{item['name']}",
                    fontName="Helvetica",
                    fontSize=8,
                    textColor=RED_ALERT if item["cds"] > 200 else AMBER_ALERT if item["cds"] > 100 else NAVY,
                    alignment=TA_CENTER,
                ),
            ),
            Paragraph(
                str(item["alerts"]),
                ParagraphStyle(
                    f"alerts_{item['name']}",
                    fontName="Helvetica-Bold" if item["alerts"] > 0 else "Helvetica",
                    fontSize=8,
                    textColor=alert_col,
                    alignment=TA_CENTER,
                ),
            ),
            Paragraph(
                item["watchlist"],
                ParagraphStyle(
                    f"watchlist_{item['name']}",
                    fontName="Helvetica-Bold",
                    fontSize=8,
                    textColor=watchlist_col,
                    alignment=TA_CENTER,
                ),
            ),
        ])

    table = Table(rows, colWidths=[4.4 * cm, 2.1 * cm, 1.6 * cm, 1.7 * cm, 1.8 * cm, 1.8 * cm, 2.6 * cm], repeatRows=1)
    table_style = [
        ("BACKGROUND", (0, 0), (-1, 0), STEEL),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [WHITE, SILVER]),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]

    for row_index, item in enumerate(sorted_portfolio, start=1):
        if item["highlight"]:
            table_style.append(("BACKGROUND", (0, row_index), (-1, row_index), HexColor("#FEF5F5")))
        if item["action"] == "Escalate":
            table_style.append(("LINEABOVE", (0, row_index), (-1, row_index), 1.0, RED_ALERT))
            table_style.append(("LINEBELOW", (0, row_index), (-1, row_index), 1.0, RED_ALERT))

    table.setStyle(TableStyle(table_style))
    return table


def render_watchlist_card(story, entry, styles):
    action_col = ACTION_COLOR.get(entry["action"], MID_GREY)
    rating_col = RATING_COLOR.get(entry["rating"], MID_GREY)

    name_style = ParagraphStyle("watchlist_name", fontName="Helvetica-Bold", fontSize=9, textColor=NAVY, leading=13)
    tag_style = ParagraphStyle("watchlist_tag", fontName="Helvetica", fontSize=8, textColor=MID_GREY, leading=11)

    header_table = Table(
        [[
            Paragraph(entry["name"], name_style),
            Paragraph(f"Score: <b>{entry['score']:.1f}</b> | {entry['type']}", tag_style),
            Paragraph(
                f"<b>{entry['rating']}</b>",
                ParagraphStyle("watchlist_rating", fontName="Helvetica-Bold", fontSize=9, textColor=rating_col, alignment=TA_CENTER),
            ),
            Paragraph(
                f"<b>{entry['action'].upper()}</b>",
                ParagraphStyle("watchlist_action", fontName="Helvetica-Bold", fontSize=8.5, textColor=action_col, alignment=TA_CENTER),
            ),
        ]],
        colWidths=[5.5 * cm, 6.0 * cm, 2.0 * cm, 2.5 * cm],
    )
    header_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), SILVER),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))

    reason_para = Paragraph(
        entry["reason"],
        ParagraphStyle(
            "watchlist_reason",
            fontName="Helvetica",
            fontSize=8,
            textColor=BODY_TEXT,
            leading=12,
            leftIndent=4,
        ),
    )

    signals_table = Table(
        [
            [Paragraph("<b>CDS</b>", styles["table_cell_bold"]), Paragraph(entry["cds"], styles["table_cell_c"])],
            [Paragraph("<b>Spread</b>", styles["table_cell_bold"]), Paragraph(entry["spread"], styles["table_cell_c"])],
            [Paragraph("<b>Eq 5d</b>", styles["table_cell_bold"]), Paragraph(entry["equity_5d"], styles["table_cell_c"])],
            [Paragraph("<b>Vol</b>", styles["table_cell_bold"]), Paragraph(entry["vol"], styles["table_cell_c"])],
        ],
        colWidths=[1.8 * cm, 1.8 * cm],
    )
    signals_table.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 2),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("ROWBACKGROUNDS", (0, 0), (-1, -1), [WHITE, SILVER]),
    ]))

    body_table = Table([[reason_para, signals_table]], colWidths=[12.2 * cm, 3.8 * cm])
    body_table.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("LEFTPADDING", (0, 0), (0, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))

    story.append(KeepTogether([header_table, body_table, sp(6)]))


def render_credit_note(story, styles, note):
    title_table = Table(
        [[
            Paragraph(
                note["name"],
                ParagraphStyle("note_name", fontName="Helvetica-Bold", fontSize=11, textColor=NAVY, leading=14),
            ),
            Paragraph(
                note["tag"],
                ParagraphStyle(
                    "note_tag",
                    fontName="Helvetica-Bold",
                    fontSize=9,
                    textColor=WHITE,
                    backColor=note["tag_color"],
                    leading=14,
                    alignment=TA_RIGHT,
                ),
            ),
        ]],
        colWidths=[CONTENT_W * 0.62, CONTENT_W * 0.38],
    )
    title_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, 0), SILVER),
        ("BACKGROUND", (1, 0), (1, 0), note["tag_color"]),
        ("TOPPADDING", (0, 0), (-1, -1), 7),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    story.append(title_table)
    story.append(sp(6))

    story.append(Paragraph(note["overview"], styles["body"]))
    story.append(sp(6))

    half_width = CONTENT_W / 2

    def strength_weak_block(items, header, header_color, body_color, width):
        content = [[Paragraph(
            header,
            ParagraphStyle("block_header", fontName="Helvetica-Bold", fontSize=8.5, textColor=WHITE, alignment=TA_CENTER),
        )]]
        for item in items:
            content.append([
                Paragraph(
                    item,
                    ParagraphStyle(
                        "block_item",
                        fontName="Helvetica",
                        fontSize=8,
                        textColor=BODY_TEXT,
                        leading=12,
                        leftIndent=4,
                        spaceAfter=2,
                    ),
                )
            ])
        table = Table(content, colWidths=[width])
        table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), header_color),
            ("BACKGROUND", (0, 1), (-1, -1), body_color),
            ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
            ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
            ("TOPPADDING", (0, 0), (-1, -1), 5),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ("LEFTPADDING", (0, 0), (-1, -1), 6),
            ("RIGHTPADDING", (0, 0), (-1, -1), 6),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ]))
        return table

    strengths_weaknesses = Table(
        [[
            strength_weak_block(note["strengths"], "STRENGTHS", GREEN_OK, HexColor("#EAF6EE"), half_width),
            strength_weak_block(note["weaknesses"], "WEAKNESSES", RED_ALERT, HexColor("#FEF5F5"), half_width),
        ]],
        colWidths=[half_width, half_width],
    )
    strengths_weaknesses.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
        ("INNERGRID", (0, 0), (-1, -1), 0, WHITE),
    ]))
    story.append(strengths_weaknesses)
    story.append(sp(8))

    sig_width = 5.5 * cm
    rating_width = 3.5 * cm
    analyst_width = CONTENT_W - sig_width - rating_width
    sig_inner_label = 3.0 * cm
    sig_inner_value = sig_width - sig_inner_label - 0.3 * cm

    signal_rows = [[Paragraph(
        "MARKET SIGNALS - 31 DEC 2025",
        ParagraphStyle("signal_header", fontName="Helvetica-Bold", fontSize=8, textColor=WHITE, alignment=TA_CENTER),
    )]]
    for label, value, color in note["signals"]:
        signal_rows.append([
            Table(
                [[
                    Paragraph(label, ParagraphStyle("signal_label", fontName="Helvetica", fontSize=8, textColor=MID_GREY)),
                    Paragraph(
                        f"<b>{value}</b>",
                        ParagraphStyle("signal_value", fontName="Helvetica-Bold", fontSize=8, textColor=color, alignment=TA_RIGHT),
                    ),
                ]],
                colWidths=[sig_inner_label, sig_inner_value],
                style=TableStyle([
                    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                    ("TOPPADDING", (0, 0), (-1, -1), 2),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
                    ("LEFTPADDING", (0, 0), (-1, -1), 0),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 0),
                ]),
            )
        ])

    signal_table = Table(signal_rows, colWidths=[sig_width])
    signal_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), STEEL),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [WHITE, SILVER]),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
    ]))

    rating_table = Table(
        [
            [Paragraph("INT. RATING", ParagraphStyle("rating_header", fontName="Helvetica-Bold", fontSize=8, textColor=WHITE, alignment=TA_CENTER))],
            [Paragraph(
                note["int_rating"],
                ParagraphStyle(
                    "rating_value",
                    fontName="Helvetica-Bold",
                    fontSize=22,
                    textColor=RATING_COLOR.get(note["int_rating"], NAVY),
                    alignment=TA_CENTER,
                ),
            )],
            [Paragraph(f"Ext: {note['ext_rating']}", ParagraphStyle("rating_ext", fontName="Helvetica", fontSize=7.5, textColor=MID_GREY, alignment=TA_CENTER))],
            [Paragraph(
                f"<b>{note['outlook']}</b>",
                ParagraphStyle(
                    "rating_outlook",
                    fontName="Helvetica-Bold",
                    fontSize=7.5,
                    textColor=RED_ALERT if "Neg" in note["outlook"] else NAVY,
                    alignment=TA_CENTER,
                ),
            )],
        ],
        colWidths=[rating_width],
    )
    rating_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), STEEL),
        ("BACKGROUND", (0, 1), (-1, -1), SILVER),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))

    analyst_table = Table(
        [
            [Paragraph("ANALYST VIEW", ParagraphStyle("analyst_header", fontName="Helvetica-Bold", fontSize=8, textColor=WHITE, alignment=TA_CENTER))],
            [Paragraph(
                note["analyst_view"],
                ParagraphStyle("analyst_body", fontName="Helvetica-Oblique", fontSize=8, textColor=BODY_TEXT, leading=12),
            )],
        ],
        colWidths=[analyst_width - 12],
    )
    analyst_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), NAVY),
        ("BACKGROUND", (0, 1), (-1, -1), HexColor("#F0F4FA")),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))

    bottom_table = Table([[signal_table, rating_table, analyst_table]], colWidths=[sig_width, rating_width, analyst_width])
    bottom_table.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
    ]))
    story.append(bottom_table)


## 7. Report assembly

The functions below define the actual report order.
Because each section is isolated, you can add, remove, or reorder sections without touching the lower-level layout code above.


In [ ]:
def add_executive_summary_section(story, styles):
    story.append(SectionHeader("01", "Executive Summary", CONTENT_W))
    story.append(sp(4))
    for bullet in EXECUTIVE_SUMMARY_BULLETS:
        story.append(bullet_para(bullet, styles))
        story.append(sp(1))
    story.append(sp(3))
    story.append(build_outlook_table())
    story.append(sp(5))


def add_portfolio_overview_section(story, styles):
    story.append(SectionHeader("02", "Portfolio Overview", CONTENT_W))
    story.append(sp(4))
    story.append(build_portfolio_table(styles))
    story.append(PageBreak())


def add_watchlist_section(story, styles):
    story.append(SectionHeader("03", "Watchlist - Key Monitored Names", CONTENT_W))
    story.append(sp(5))
    story.append(Paragraph(WATCHLIST_INTRO, styles["body"]))
    story.append(sp(5))

    story.append(Paragraph(
        "ESCALATE - Immediate Committee Action Required",
        ParagraphStyle(
            "watchlist_escalate_header",
            fontName="Helvetica-Bold",
            fontSize=9,
            textColor=WHITE,
            backColor=RED_ALERT,
            leftIndent=5,
            rightIndent=5,
            leading=14,
            spaceBefore=4,
            spaceAfter=6,
        ),
    ))
    for entry in [item for item in WATCHLIST_ENTRIES if item["action"] == "Escalate"]:
        render_watchlist_card(story, entry, styles)

    story.append(sp(4))
    story.append(Paragraph(
        "REVIEW - Timely Analytical Action Required",
        ParagraphStyle(
            "watchlist_review_header",
            fontName="Helvetica-Bold",
            fontSize=9,
            textColor=WHITE,
            backColor=AMBER_ALERT,
            leftIndent=5,
            rightIndent=5,
            leading=14,
            spaceBefore=4,
            spaceAfter=4,
        ),
    ))
    for entry in [item for item in WATCHLIST_ENTRIES if item["action"] == "Review"]:
        render_watchlist_card(story, entry, styles)

    story.append(PageBreak())


def add_detailed_credit_notes_section(story, styles):
    story.append(SectionHeader("04", "Detailed Credit Notes", CONTENT_W))
    story.append(sp(5))
    story.append(Paragraph(
        "In-depth assessments for one bank and one sovereign, selected on analytical significance and committee relevance as of the reference date.",
        styles["body"],
    ))
    story.append(sp(6))

    for index, note in enumerate(DETAILED_NOTES):
        render_credit_note(story, styles, note)
        if index < len(DETAILED_NOTES) - 1:
            story.append(sp(10))

    story.append(PageBreak())


def add_market_monitoring_section(story, styles):
    story.append(SectionHeader("05", "Market Monitoring Insights", CONTENT_W))
    story.append(sp(5))

    rows = [[Paragraph("<b>Theme</b>", styles["table_header"]), Paragraph("<b>Key Observations</b>", styles["table_header"])]]
    for theme, observation in MARKET_MONITORING_ROWS:
        rows.append([
            Paragraph(f"<b>{theme}</b>", styles["table_cell_bold"]),
            Paragraph(observation, styles["table_cell"]),
        ])

    table = Table(rows, colWidths=[3.5 * cm, 12.5 * cm], repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), STEEL),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [WHITE, SILVER]),
        ("BOX", (0, 0), (-1, -1), 0.5, LINE_GREY),
        ("INNERGRID", (0, 0), (-1, -1), 0.3, LINE_GREY),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    story.append(table)
    story.append(sp(3))
    story.append(hr())
    story.append(sp(5))


def add_methodology_section(story, styles):
    story.append(SectionHeader("06", "Methodology", CONTENT_W))
    story.append(sp(5))
    for bullet in METHODOLOGY_BULLETS:
        story.append(bullet_para(bullet, styles))
        story.append(sp(3))
    story.append(sp(8))
    story.append(hr(thickness=0.3))
    story.append(sp(5))
    story.append(Paragraph(DISCLAIMER_TEXT, styles["disclaimer"]))


def build_report():
    styles = build_styles()
    story = [PageBreak()]
    add_executive_summary_section(story, styles)
    add_portfolio_overview_section(story, styles)
    add_watchlist_section(story, styles)
    add_detailed_credit_notes_section(story, styles)
    add_market_monitoring_section(story, styles)
    add_methodology_section(story, styles)
    return story


## 8. Generate the PDF

Run this last cell after any edits.
It rebuilds the full report and writes the PDF into the project root.


In [ ]:
doc = SimpleDocTemplate(
    str(OUTPUT_PATH),
    pagesize=A4,
    leftMargin=MARGIN_L,
    rightMargin=MARGIN_R,
    topMargin=MARGIN_T + 0.55 * cm,
    bottomMargin=MARGIN_B + 0.35 * cm,
    title="Credit Risk Monitoring Report - Internal",
    author="Thanh Pham | GCH Credit Risk Division",
    subject=f"Internal Credit Risk Committee Report - {REPORT_CONFIG['reference_date']}",
)

story = build_report()
doc.build(story, onFirstPage=draw_cover_page, onLaterPages=draw_page_frame)
print(f"Report generated: {OUTPUT_PATH}")
